[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nepslor/B5203E-TSAF/blob/main/W8/probabilistic_forecasts.ipynb)

# Probabilistic forecasts: bootstrap, quantile loss and scores
A point forecast tells us what to expect. A probabilistic forecast also tells us how wrong we could be. In this exercise we forecast the hourly temperature in Jena for the next 24 hours, and we build the distribution of the forecast in two ways:
* **bootstrap**: we fit a point forecaster, look at the errors it makes on data it has never seen, and reuse them as scenarios or as quantiles
* **quantile loss**: we fit one model for each quantile directly

We then compare the two with the quantile score and the reliability. The notebook contains three exercises:
1. bootstrap for scenarios
2. from errors to quantile forecasts
3. quantile crossing and probabilistic scores

We use the Jena climate dataset, recorded by the weather station of the Max Planck Institute for Biogeochemistry in Jena (Germany) from 2009 to 2016 at 10 minutes resolution. We only keep the temperature and resample it to 1 hour.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from lightgbm import LGBMRegressor

In [ ]:
url = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip'
df = pd.read_csv(url, compression='zip')
df.index = pd.to_datetime(df.pop('Date Time'), format='%d.%m.%Y %H:%M:%S')

# hourly temperature. A few hours are missing: we leave them as NaN and drop them later
T = df['T (degC)'].resample('1h').mean()
print('observations: {}, missing: {}'.format(len(T), T.isna().sum()))
T.iloc[-24*21:].plot(figsize=(15, 3), ylabel='T (degC)');

## Features, targets and split
We use a direct strategy: at time $t$ the features are the last 24 values of the temperature plus the hour and the day of the year, and the targets are the next `n_sa = 24` values. Each row of `x` and `y` is one forecast origin.

We split the data chronologically in three blocks:
* **training** set: used to fit the models
* **calibration** set: never seen by the models, used to measure the errors they make on new data
* **test** set: used only to score the final probabilistic forecasts

We leave `n_sa` rows between two blocks, so that no target value appears in two of them.

In [ ]:
n_sa = 24     # steps ahead to be predicted
n_lags = 24   # past values used as features

x = pd.concat({'lag_{}'.format(l): T.shift(l) for l in range(n_lags)}, axis=1)
x['hour'], x['dayofyear'] = x.index.hour, x.index.dayofyear
y = pd.concat({'step_{}'.format(h): T.shift(-h) for h in range(1, n_sa + 1)}, axis=1)

# keep only the forecast origins with complete features and targets
keep = x.notna().all(axis=1) & y.notna().all(axis=1)
x, y = x[keep], y[keep]

# chronological split: train | calibration | test, one year each for the last two
n_cal = n_te = 24 * 365
i_cal, i_te = len(x) - n_te - n_sa - n_cal, len(x) - n_te
x_tr, y_tr = x.iloc[:i_cal - n_sa], y.iloc[:i_cal - n_sa]
x_cal, y_cal = x.iloc[i_cal:i_te - n_sa], y.iloc[i_cal:i_te - n_sa]
x_te, y_te = x.iloc[i_te:], y.iloc[i_te:]

for name, x_i in zip(['train', 'cal', 'test'], [x_tr, x_cal, x_te]):
  print('{:>5}: {} rows, from {} to {}'.format(name, len(x_i), x_i.index[0], x_i.index[-1]))

## Point forecaster
The point forecaster is a collection of boosted trees, one for each step ahead, fitted on the training set.

In [ ]:
# n_jobs=2: on a dataset this small, more threads only add overhead
lgb_pars = dict(force_col_wise=True, verbose=-1, n_jobs=2)
lgb_models = [LGBMRegressor(**lgb_pars).fit(x_tr, y_tr[c].values)
              for c in tqdm(y_tr.columns, desc='fitting LGBM')]

# predictions of a list of models, one column for each step ahead
predict = lambda models, x: np.vstack([m.predict(x) for m in models]).T

y_hat_cal, y_hat_te = predict(lgb_models, x_cal), predict(lgb_models, x_te)
print('y_hat_cal: {}, y_hat_te: {}'.format(y_hat_cal.shape, y_hat_te.shape))

# ❓ 1. Bootstrap for scenarios
The errors of the forecaster on the calibration set form a matrix

$$\varepsilon_{t, h} = y_{t+h} - \hat{y}_{t+h \mid t}$$

with one row for each forecast origin $t$ and one column for each step ahead $h$. Each row is an error trajectory that the model really produced.

The idea of the bootstrap is that the errors the model will make tomorrow look like the ones it made in the past. We draw rows of the error matrix at random, with replacement, and add them to a new point forecast. Each draw is a **scenario**:

$$\hat{y}^{(s)}_{t+h \mid t} = \hat{y}_{t+h \mid t} + \varepsilon_{t_s, h} \qquad t_s \sim \mathcal{U}\{1, \dots, n_{cal}\}$$

1. Complete the function `bootstrap_scenarios`.
2. The function `independent_scenarios` draws a different row for each step ahead. Compare the two sets of scenarios in the plots below. At each step ahead they have the same distribution: what is different?
3. Look at the histogram of the mean temperature over the next 24 hours. Which of the two samplings would you trust to estimate the energy needed to heat a building tomorrow?

In [ ]:
rng = np.random.default_rng(0)
err_cal = y_cal.values - y_hat_cal   # (n_cal, n_sa): one error trajectory per row

def bootstrap_scenarios(y_hat, errs, n_scenarios=100):
  # y_hat: (n_sa,) point forecast, errs: (n, n_sa) error matrix
  # draw n_scenarios rows of errs at random, with replacement, and add them to y_hat
  idx = ?
  return ?

def independent_scenarios(y_hat, errs, n_scenarios=100):
  # same as above, but a different row is drawn for each step ahead
  idx = rng.integers(0, len(errs), (n_scenarios, errs.shape[1]))
  return y_hat + errs[idx, np.arange(errs.shape[1])]

In [ ]:
i = 24 * 200   # a forecast origin in the test set
scen = bootstrap_scenarios(y_hat_te[i], err_cal)
scen_ind = independent_scenarios(y_hat_te[i], err_cal)

fig, ax = plt.subplots(1, 3, figsize=(18, 4))
titles = ['bootstrap: whole error trajectories', 'independent draw at each step']
for a, s, title in zip(ax, [scen, scen_ind], titles):
  a.plot(s.T, color='r', alpha=0.1)
  a.plot(y_hat_te[i], color='orange', lw=2, label='y_hat')
  a.plot(y_te.values[i], color='b', lw=2, label='y_te')
  a.set(title=title, xlabel='step ahead [h]', ylabel='T (degC)')
ax[0].legend()

# distribution of the mean temperature over the next 24 hours, using more scenarios
means = {'bootstrap': bootstrap_scenarios(y_hat_te[i], err_cal, 5000).mean(axis=1),
         'independent': independent_scenarios(y_hat_te[i], err_cal, 5000).mean(axis=1)}
for k, m in means.items():
  ax[2].hist(m, bins=50, alpha=0.5, density=True, label='{}, std={:0.2f}'.format(k, m.std()))
ax[2].axvline(y_te.values[i].mean(), color='b', label='observed')
ax[2].set(title='mean T over the next 24h', xlabel='T (degC)')
ax[2].legend();

# ❓ 2. From errors to quantiles
Scenarios describe whole trajectories. Often we only need the distribution of the forecast at each step ahead, summarized by a few quantiles. If we drew infinitely many bootstrap scenarios and computed their quantiles at step $h$, we would obtain the point forecast plus the quantiles of the column $h$ of the error matrix. So we don't need to sample at all, and two small functions are enough:
* `retrieve_quantiles` computes the quantiles of the errors for each step ahead
* `predict_quantiles` adds them to the point forecasts

`q_vect` is the usual grid `np.arange(11)/10` with the two extremes moved from 0 and 1 to 0.01 and 0.99: the minimum and the maximum of the errors are poorly estimated, and the quantile loss of LightGBM that we use later requires $0 < \tau < 1$.

1. Use the two functions to obtain `q_boot`, the quantile forecasts on the test set, with shape `(n_te, n_sa, len(q_vect))`.
2. On which set do you compute the quantiles of the errors? Why not on the training set?
3. Run the animation. How does the width of the intervals change with the step ahead? And with the forecast origin?

In [ ]:
q_vect = np.clip(np.arange(11)/10, 0.01, 0.99)   # [0.01, 0.1, 0.2, ..., 0.9, 0.99]

def retrieve_quantiles(y, y_hat, q_vect=q_vect):
  err = y - y_hat
  # retrieve quantiles of the errors, one row for each step ahead
  q_errs = np.quantile(err, q_vect, axis=0).T
  return q_errs

def predict_quantiles(y_hat, q_errs):
  # sum quantiles to the pointwise predictions
  return np.expand_dims(y_hat, 2) + np.expand_dims(q_errs, 0)

q_errs = ?
q_boot = ?

ax = pd.DataFrame(q_errs, columns=q_vect).plot(figsize=(8, 3), cmap='coolwarm', xlabel='step ahead [h]', ylabel='error quantiles (degC)')
ax.legend(title='tau', fontsize='x-small', loc='center left', bbox_to_anchor=(1, 0.5));

The following cell downloads the plot function `qs_animation_plotly` from the course repository. It animates observations, point forecasts and quantile forecasts over consecutive forecast origins: each band is the interval between two symmetric quantiles.

In [ ]:
#@title  Fast Quantile Plot (imported from GitHub)
import importlib.util
from pathlib import Path

import requests

# Keeps the notebook standalone in Colab by downloading the helper module at runtime.
PLOT_UTIL_URL = "https://raw.githubusercontent.com/nepslor/B5203E-TSAF/main/utils/plot_util.py"
PLOT_UTIL_PATH = Path("plot_util.py")

resp = requests.get(PLOT_UTIL_URL, timeout=30)
resp.raise_for_status()
PLOT_UTIL_PATH.write_bytes(resp.content)

spec = importlib.util.spec_from_file_location("plot_util", PLOT_UTIL_PATH)
plot_util = importlib.util.module_from_spec(spec)
spec.loader.exec_module(plot_util)

qs_animation_plotly = plot_util.qs_animation_plotly

In [ ]:
skip, n_plots = 24 * 200, 200
w = slice(skip, skip + n_plots)
display(qs_animation_plotly(y_te.values[w], y_hat_te[w], q_boot[w], n_rows=n_plots, f_name='LGBM + bootstrap'))

# Quantile loss
Instead of fitting the mean and adding its errors, we can fit each quantile directly. The quantile $\tau$ of $y$ given $x$ is the minimizer of the quantile loss, also called pinball loss:

$$\rho_{\tau}(y, \hat{q}) = \left(\mathbb{1}_{y < \hat{q}} - \tau\right)\left(\hat{q} - y\right)$$

For $\tau = 0.9$ a forecast below the observation costs 9 times more than a forecast above it by the same amount, so the minimizer is pushed up until only 10% of the observations are above it.

In LightGBM we just need to set `objective='quantile'` and `alpha=tau`. We fit one model for each quantile and for each step ahead, that is `len(q_vect) * n_sa = 264` independent models: this takes a couple of minutes. These models are fitted on the training set only and don't need the calibration set.

In [ ]:
lgb_models_q = [[LGBMRegressor(objective='quantile', alpha=tau, **lgb_pars).fit(x_tr, y_tr[c].values)
                 for c in y_tr.columns] for tau in tqdm(q_vect, desc='fitting quantile LGBM')]

# quantile forecasts on the test set: (n_te, n_sa, len(q_vect)). The median is our point forecast
q_lgb = np.dstack([predict(models, x_te) for models in lgb_models_q])
y_hat_te_q = q_lgb[:, :, q_vect == 0.5][:, :, 0]
print('q_lgb: {}'.format(q_lgb.shape))

Run the animation for the new forecasts and compare it with the previous one. Do you notice anything strange in the bands?

In [ ]:
display(qs_animation_plotly(y_te.values[w], y_hat_te_q[w], q_lgb[w], n_rows=n_plots, f_name='quantile LGBM'))

# ❓ 3. Quantile crossing and probabilistic scores
The quantiles of a distribution are ordered: $q_{0.1} \leq q_{0.2} \leq \dots \leq q_{0.9}$. The bootstrap quantiles respect this order by construction, since we add sorted error quantiles to the same point forecast. The quantile models instead were fitted independently, and nothing forces them to agree with each other. When a lower quantile is above a higher one we have a **quantile crossing**.

1. Complete `crossing`, which returns the fraction of (forecast origin, step ahead) pairs in which at least two consecutive quantiles cross. Evaluate it on `q_boot` and `q_lgb`.
2. A simple fix is to sort the forecasts along the quantile axis. Use it to build `q_lgb_sorted`.

In [ ]:
# fraction of (forecast origin, step ahead) pairs with at least one crossing
crossing = lambda q: ?
print('crossing bootstrap: {:0.3f}, crossing quantile LGBM: {:0.3f}'.format(crossing(q_boot), crossing(q_lgb)))

q_lgb_sorted = ?

The next cell plots the forecast with the largest crossing, before and after sorting.

In [ ]:
i = np.argmin(np.diff(q_lgb, axis=2).min(axis=(1, 2)))   # forecast origin with the largest crossing
fig, ax = plt.subplots(1, 2, figsize=(15, 4), sharey=True)
for a, q, title in zip(ax, [q_lgb[i], q_lgb_sorted[i]], ['quantile LGBM', 'quantile LGBM sorted']):
  for j, tau in enumerate(q_vect):
    a.plot(q[:, j], color=plt.cm.coolwarm(tau), label=tau)
  a.plot(y_te.values[i], 'k--', label='y_te')
  a.set(title='{}, origin: {}'.format(title, y_te.index[i]), xlabel='step ahead [h]', ylabel='T (degC)')
ax[1].legend(title='tau', fontsize='x-small', loc='center left', bbox_to_anchor=(1, 0.5));

## Probabilistic scores
We now have three probabilistic forecasts for the test set. We compare them with two scores, both computed for each quantile $\tau$:
* the **quantile score** is the average quantile loss $\rho_{\tau}$ of the forecasted quantile, over all the forecast origins and steps ahead. Lower is better. Its integral over $\tau$ is half of the CRPS, so its mean over `q_vect` is a coarse approximation of it.
* the **reliability** is the fraction of observations that fall below the forecasted quantile. For a reliable forecast this fraction is $\tau$ itself, so the curve lies on the diagonal.

3. Use `quantile_score` and `reliability` to score the three forecasts on the test set, then answer:
    * does sorting change the scores?
    * which forecast is more reliable? Which one has the lowest quantile score? Is one of them clearly better?
    * the quantile score barely notices the crossings. Why are they still a problem?

In [ ]:
def quantile_score(q, y, q_vect=np.arange(11)/10):
  qs = {}
  for i, tau in enumerate(q_vect):
    err_tau = q[:, :, i] - y
    I = (err_tau > 0).astype(int)
    qs[tau] = np.mean((I - tau) * err_tau)
  return qs

def reliability(q, y, q_vect=np.arange(10)/10):
  res = {}
  for i, tau in enumerate(q_vect):
    res[tau] = np.mean(y < q[:, :, i])
  return res

forecasts = {'LGBM + bootstrap': q_boot, 'quantile LGBM': q_lgb, 'quantile LGBM sorted': q_lgb_sorted}

# one column for each forecast, one row for each quantile. Remember to pass q_vect to the score functions
qscore = pd.DataFrame({k: ? for k, q in forecasts.items()})
rel = pd.DataFrame({k: ? for k, q in forecasts.items()})

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
qscore.plot(ax=ax[0], title='quantile score', xlabel='tau', style=['-', '-', '--'])
rel.plot(ax=ax[1], title='reliability', xlabel='tau', style=['-', '-', '--'])
ax[1].plot([0, 1], [0, 1], 'k:')
print('mean quantile score:\n{}'.format(qscore.mean().round(4)))

In [ ]:
# quantile score and reliability minus tau (0 for a perfectly reliable forecast)
pd.concat({'quantile score': qscore, 'reliability - tau': rel.sub(rel.index, axis=0)}, axis=1).round(3)